# DemandPulse
## Notebook 03 — Causal Promotion Impact Analysis

**Objective:** estimate whether promotional activity (`activity_flag`) changes observed sales after adjustment for measured pre-treatment differences.

This notebook is a **standalone observational causal-inference study**. It does not rely on live Python state from Notebook 01 or Notebook 02. It preserves the same remote Hugging Face access pattern, complete-series SHA-256 sampling rule, and leakage-safe historical features established earlier in DemandPulse.

Primary questions:

- Do promotions increase observed sales?
- What are the Average Treatment Effect (ATE) and Average Treatment Effect on the Treated (ATT)?
- How stable are estimates across adjustment methods and sensitivity analyses?
- Which product/store segments appear more or less responsive?
- Do overlap, placebo tests, stockout censoring, or model sensitivity weaken causal credibility?

> **Scientific caution:** FreshRetailNet-50K is observational. The estimates below are causal only under stated assumptions such as consistency, positivity, correct temporal ordering, and no important unmeasured confounding. They do not prove causality with experimental certainty.

## 1. Environment

In [ ]:
%pip install -q "datasets>=3.0" "pandas>=2.0" "numpy>=1.24" "pyarrow>=14" "matplotlib>=3.7" "scikit-learn>=1.3" "lightgbm>=4.0" "tqdm>=4.66"

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import sys
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import datasets
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from datasets import load_dataset
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f"Python      : {sys.version.split()[0]}")
print(f"datasets    : {datasets.__version__}")
print(f"pandas      : {pd.__version__}")
print(f"numpy       : {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"LightGBM    : {lgb.__version__}")

## 2. Project configuration

In [ ]:
DATASET_ID = "Dingdong-Inc/FreshRetailNet-50K"
SERIES_SAMPLE_PERCENT = 30
SERIES_KEYS = ["store_id", "product_id"]
STREAM_BATCH_SIZE = 50_000

N_SPLITS = 5
N_BOOTSTRAPS = 500
DEBUG_N_BOOTSTRAPS = 100
RUN_FULL_BOOTSTRAP = True
BOOTSTRAP_REPS = N_BOOTSTRAPS if RUN_FULL_BOOTSTRAP else DEBUG_N_BOOTSTRAPS

PROPENSITY_EPS = 0.01
POOR_OVERLAP_EXTREME_PCT = 5.0
MIN_SEGMENT_ROWS = 1_000
MIN_SEGMENT_TREATED = 100
MIN_SEGMENT_CONTROL = 100
SEGMENT_REFERENCE_WINDOW = 28

REPORTS_DIR = Path("reports")
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = Path("models")
CAUSAL_MODELS_DIR = MODELS_DIR / "causal"
for directory in (REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, CAUSAL_MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

assert 0 < SERIES_SAMPLE_PERCENT <= 100
assert N_SPLITS >= 2
assert 0 < PROPENSITY_EPS < 0.5
assert BOOTSTRAP_REPS >= 50

print(f"Dataset                 : {DATASET_ID}")
print(f"Complete-series sample  : {SERIES_SAMPLE_PERCENT}%")
print(f"Cross-fitting folds      : {N_SPLITS}")
print(f"Cluster bootstraps       : {BOOTSTRAP_REPS}")

## 3. Treatment, outcome, causal roles, and the discount rule

Primary treatment and outcome:

- `T = activity_flag`
- `T = 1`: promotional/activity treatment
- `T = 0`: no promotional activity
- `Y = sale_amount`

`discount_pct = 1 - discount` is created for **descriptive treatment characterization**, but same-day discount is **not included in the primary propensity or outcome-adjustment feature set**. A plausible mechanism is:

```text
Promotion ──→ Discount ──→ Sales
```

If promotion changes discount, discount is a mediator/post-treatment variable. Conditioning on it can block part of the total promotion effect. A mediator-adjusted model therefore answers a different question and is not the primary estimand here.

### Causal vocabulary

- **Treatment:** exposure whose effect is being estimated (`activity_flag`).
- **Outcome:** response of interest (`sale_amount`).
- **Confounder:** pre-treatment cause/predictor of both treatment assignment and outcome.
- **Mediator:** variable on a causal path from treatment to outcome; adjusting for it can remove part of the total effect.
- **Collider:** variable caused by two other variables; conditioning on it can create spurious association.

The primary analysis deliberately avoids obvious same-day post-treatment variables.

## 4. Conceptual causal DAG

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.axis("off")

nodes = {
    "Historical\nDemand": (0.10, 0.78),
    "Holiday": (0.10, 0.58),
    "Store /\nProduct": (0.10, 0.38),
    "Seasonality": (0.10, 0.18),
    "Weather": (0.45, 0.08),
    "Promotion": (0.48, 0.58),
    "Discount\n(mediator?)": (0.70, 0.58),
    "Sales": (0.90, 0.38),
}

for name, (x, y) in nodes.items():
    ax.text(x, y, name, ha="center", va="center", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black"))

def arrow(a, b, label=None):
    x1, y1 = nodes[a]; x2, y2 = nodes[b]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=1.4))
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2 + 0.025, label, fontsize=8, ha="center")

for src in ["Historical\nDemand", "Holiday", "Store /\nProduct", "Seasonality"]:
    arrow(src, "Promotion")
    arrow(src, "Sales")
arrow("Weather", "Sales")
arrow("Promotion", "Sales", "total path")
arrow("Promotion", "Discount\n(mediator?)")
arrow("Discount\n(mediator?)", "Sales")

ax.set_title("Conceptual DAG for Promotion and Observed Sales", fontsize=13)
plt.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIGURES_DIR / "causal_dag.png", dpi=160, bbox_inches="tight")
plt.show()

The DAG is conceptual, not a claim that every edge is known to be true. Historical demand, holiday timing, store/product identity, and seasonality are treated as important measured pre-treatment adjustment variables. Weather is assumed not to be caused by promotion and may improve outcome precision; the notebook explicitly repeats the analysis without weather.

## 5. Remote data access, schema validation, and deterministic series sampling

In [ ]:
REQUIRED_SOURCE_COLUMNS = [
    "city_id", "store_id", "management_group_id", "first_category_id",
    "second_category_id", "third_category_id", "product_id", "dt",
    "sale_amount", "hours_sale", "stock_hour6_22_cnt", "hours_stock_status",
    "discount", "holiday_flag", "activity_flag", "precpt",
    "avg_temperature", "avg_humidity", "avg_wind_level",
]

# Sequence fields are intentionally not materialized: same-day and memory-heavy.
MATERIALIZED_COLUMNS = [
    "city_id", "store_id", "management_group_id", "first_category_id",
    "second_category_id", "third_category_id", "product_id", "dt",
    "sale_amount", "stock_hour6_22_cnt", "discount", "holiday_flag",
    "activity_flag", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level",
]

train_stream = load_dataset(DATASET_ID, split="train", streaming=True)
eval_stream = load_dataset(DATASET_ID, split="eval", streaming=True)

stream_columns = list(train_stream.features.keys()) if train_stream.features is not None else []
missing_source = sorted(set(REQUIRED_SOURCE_COLUMNS) - set(stream_columns))
assert not missing_source, f"Remote train schema is missing required columns: {missing_source}"

print(type(train_stream))
print("Schema verified. Columns:")
print(stream_columns)

In [ ]:
def series_selected(
    store_id: object,
    product_id: object,
    percent: float,
    cache: Dict[Tuple[object, object], bool],
) -> bool:
    """Notebook-01-compatible stable SHA-256 complete-series selection rule."""
    if percent >= 100:
        return True
    key = (store_id, product_id)
    if key not in cache:
        token = f"{store_id}-{product_id}".encode("utf-8")
        digest = hashlib.sha256(token).digest()
        bucket = int.from_bytes(digest[:8], byteorder="big", signed=False) % 10_000
        cache[key] = bucket < int(round(percent * 100))
    return cache[key]


def materialize_train_stream(
    stream,
    sample_percent: float,
    batch_size: int = STREAM_BATCH_SIZE,
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """Stream, deterministic-series filter, and materialize selected train columns once."""
    if not hasattr(stream, "batch"):
        raise RuntimeError(
            "Expected a recent Hugging Face IterableDataset with .batch(). "
            "Re-run the install cell and restart the kernel."
        )

    frames: List[pd.DataFrame] = []
    selection_cache: Dict[Tuple[object, object], bool] = {}
    rows_processed = 0
    rows_selected = 0

    for batch in tqdm(stream.batch(batch_size=batch_size), desc="Streaming train", unit="batch"):
        batch_df = pd.DataFrame(batch)
        missing = sorted(set(REQUIRED_SOURCE_COLUMNS) - set(batch_df.columns))
        if missing:
            raise AssertionError(f"A streamed train batch is missing columns: {missing}")

        rows_processed += len(batch_df)
        keep = np.fromiter(
            (series_selected(s, p, sample_percent, selection_cache)
             for s, p in zip(batch_df["store_id"], batch_df["product_id"])),
            dtype=bool,
            count=len(batch_df),
        )
        selected = batch_df.loc[keep, MATERIALIZED_COLUMNS].copy()
        rows_selected += len(selected)
        if not selected.empty:
            frames.append(selected)

    if not frames:
        raise RuntimeError("Deterministic series sampling selected zero train rows.")

    out = pd.concat(frames, ignore_index=True, copy=False)
    stats = {"rows_processed": rows_processed, "rows_selected": rows_selected}
    return out, stats


def collect_sampled_series_ids(stream, sample_percent: float, batch_size: int = STREAM_BATCH_SIZE) -> set:
    """Collect only sampled eval series identities; no eval outcomes are used in causal estimation."""
    if not hasattr(stream, "batch"):
        raise RuntimeError("Expected IterableDataset.batch() support.")
    cache: Dict[Tuple[object, object], bool] = {}
    series = set()
    for batch in tqdm(stream.batch(batch_size=batch_size), desc="Scanning eval series IDs", unit="batch"):
        stores = batch["store_id"]
        products = batch["product_id"]
        for s, p in zip(stores, products):
            if series_selected(s, p, sample_percent, cache):
                series.add((s, p))
    return series

train_df, materialization_stats = materialize_train_stream(train_stream, SERIES_SAMPLE_PERCENT)
eval_series_ids = collect_sampled_series_ids(eval_stream, SERIES_SAMPLE_PERCENT)

train_df["dt"] = pd.to_datetime(train_df["dt"], errors="raise")
print(materialization_stats)
print("train shape:", train_df.shape)
print("memory MiB:", round(train_df.memory_usage(deep=True).sum() / 1024**2, 2))
print("eval sampled series identities:", len(eval_series_ids))

The eval split is scanned **only for sampled `(store_id, product_id)` membership** so an “existing-series-only” sensitivity analysis can preserve the project’s previous series definition. Eval outcomes, promotions, discounts, weather, and sales are not used in causal model fitting, effect estimation, tuning, or calibration.

## 6. Data validation and causal data audit

In [ ]:
missing_materialized = sorted(set(MATERIALIZED_COLUMNS) - set(train_df.columns))
assert not missing_materialized, f"Materialized train data missing columns: {missing_materialized}"

key_cols = SERIES_KEYS + ["dt"]
duplicate_mask = train_df.duplicated(key_cols, keep=False)
if duplicate_mask.any():
    display(train_df.loc[duplicate_mask].sort_values(key_cols).head(50))
    raise AssertionError(
        f"Found {duplicate_mask.sum():,} rows involved in duplicate (store_id, product_id, dt) keys. "
        "Duplicates are displayed above and are not silently removed."
    )

assert train_df["sale_amount"].notna().all(), "sale_amount contains missing values."
assert (train_df["sale_amount"] >= 0).all(), "sale_amount contains negative values."

observed_treatments = set(pd.Series(train_df["activity_flag"]).dropna().unique().tolist())
assert observed_treatments.issubset({0, 1}), f"activity_flag must be binary; observed {observed_treatments}"
assert train_df["activity_flag"].notna().all(), "activity_flag contains missing values."

train_df["T"] = train_df["activity_flag"].astype("int8")
train_df["Y"] = train_df["sale_amount"].astype(float)
train_df["discount_pct"] = 1.0 - pd.to_numeric(train_df["discount"], errors="coerce")
train_df["_series_id"] = train_df["store_id"].astype(str) + "::" + train_df["product_id"].astype(str)
train_df["existing_series"] = [
    (s, p) in eval_series_ids for s, p in zip(train_df["store_id"], train_df["product_id"])
]

missing_table = pd.DataFrame({
    "column": train_df.columns,
    "missing_count": [int(train_df[c].isna().sum()) for c in train_df.columns],
    "missing_percentage": [float(train_df[c].isna().mean() * 100) for c in train_df.columns],
    "dtype": [str(train_df[c].dtype) for c in train_df.columns],
}).sort_values("missing_percentage", ascending=False)

audit_metrics = {
    "total_rows": len(train_df),
    "treatment_rows": int(train_df["T"].sum()),
    "control_rows": int((1 - train_df["T"]).sum()),
    "treatment_percentage": float(train_df["T"].mean() * 100),
    "control_percentage": float((1 - train_df["T"]).mean() * 100),
    "unique_stores": int(train_df["store_id"].nunique()),
    "unique_products": int(train_df["product_id"].nunique()),
    "unique_series": int(train_df["_series_id"].nunique()),
    "date_min": train_df["dt"].min(),
    "date_max": train_df["dt"].max(),
    "duplicate_key_rows": int(duplicate_mask.sum()),
    "negative_sales_rows": int((train_df["Y"] < 0).sum()),
    "zero_sales_percentage": float((train_df["Y"] == 0).mean() * 100),
    "existing_series_rows_percentage": float(train_df["existing_series"].mean() * 100),
}
causal_data_audit = pd.DataFrame({"metric": list(audit_metrics.keys()), "value": list(audit_metrics.values())})
display(causal_data_audit)
display(missing_table.head(20))
causal_data_audit.to_csv(TABLES_DIR / "causal_data_audit.csv", index=False)

## 7. Pre-treatment feature engineering

In [ ]:
def build_pretreatment_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create only treatment-time-available historical and calendar covariates."""
    out = df.sort_values(SERIES_KEYS + ["dt"]).copy()
    g = out.groupby(SERIES_KEYS, sort=False, observed=True)

    for lag in (1, 7, 14, 28):
        out[f"lag_{lag}"] = g["Y"].shift(lag)

    shifted_sales = g["Y"].shift(1)
    out["_sales_shift1"] = shifted_sales
    sg = out.groupby(SERIES_KEYS, sort=False, observed=True)["_sales_shift1"]

    for window in (7, 28):
        out[f"rolling_mean_{window}"] = sg.transform(
            lambda s, w=window: s.rolling(w, min_periods=max(2, w // 4)).mean()
        )
        out[f"rolling_std_{window}"] = sg.transform(
            lambda s, w=window: s.rolling(w, min_periods=max(2, w // 4)).std()
        )

    # Intermittency is also historical: fraction of zero sales in the prior 28 observations.
    out["_zero_shift1"] = g["Y"].shift(1).eq(0).astype(float)
    zg = out.groupby(SERIES_KEYS, sort=False, observed=True)["_zero_shift1"]
    out["historical_zero_ratio_28"] = zg.transform(
        lambda s: s.rolling(28, min_periods=7).mean()
    )

    # Current-day stock information is NOT a predictor. Only lagged history is retained.
    out["_stockout_today"] = pd.to_numeric(out["stock_hour6_22_cnt"], errors="coerce").fillna(0).gt(0).astype("int8")
    stock_g = out.groupby(SERIES_KEYS, sort=False, observed=True)["_stockout_today"]
    out["historical_stockout_lag_1"] = stock_g.shift(1)
    out["historical_stockout_lag_7"] = stock_g.shift(7)
    out["recent_stockout_history"] = (
        out[["historical_stockout_lag_1", "historical_stockout_lag_7"]].fillna(0).max(axis=1) > 0
    ).astype("int8")

    out["day_of_week"] = out["dt"].dt.dayofweek.astype("int8")
    out["month"] = out["dt"].dt.month.astype("int8")
    out["is_weekend"] = out["day_of_week"].isin([5, 6]).astype("int8")

    out.drop(columns=["_sales_shift1", "_zero_shift1"], inplace=True)
    return out

feature_df = build_pretreatment_features(train_df)

# The primary estimand is restricted to observations with enough prior history for lag_28.
history_required = ["lag_1", "lag_7", "lag_14", "lag_28", "rolling_mean_7", "rolling_mean_28"]
pre_history_rows = len(feature_df)
analytic_df = feature_df.dropna(subset=history_required).copy()
print(f"Rows before history eligibility: {pre_history_rows:,}")
print(f"Rows after history eligibility : {len(analytic_df):,}")
print(f"Rows removed                   : {pre_history_rows - len(analytic_df):,}")

# Sanity checks for feature availability and numerical integrity.
assert not analytic_df["Y"].isna().any()
assert (analytic_df["dt"] == analytic_df["dt"]).all()
for col in history_required:
    assert np.isfinite(pd.to_numeric(analytic_df[col], errors="coerce").dropna()).all(), f"Infinite values in {col}"

# Training-derived, pre-treatment segmentation variables.
q_low, q_high = analytic_df["rolling_mean_28"].quantile([1/3, 2/3]).tolist()
analytic_df["demand_level"] = pd.cut(
    analytic_df["rolling_mean_28"],
    bins=[-np.inf, q_low, q_high, np.inf],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)
analytic_df["intermittency_level"] = pd.cut(
    analytic_df["historical_zero_ratio_28"],
    bins=[-np.inf, 1/3, 2/3, np.inf],
    labels=["Low Intermittency", "Medium Intermittency", "High Intermittency"],
    include_lowest=True,
)

# Release superseded materializations before the ML stages.
del feature_df, train_df
gc.collect()

### Leakage audit

The primary nuisance models never receive current-day `sale_amount`, `hours_sale`, `hours_stock_status`, or `stock_hour6_22_cnt`. Sales lags and rolling statistics are shifted first. Historical stockout features are lagged. `discount`/`discount_pct` are excluded from the primary adjustment set because they may lie downstream of promotion.

Same-day weather may enter the retrospective primary model under the assumption that promotion does not cause weather. A sensitivity analysis repeats the full cross-fitted AIPW estimation without weather.

## 8. Raw treated/control audit — associations only

In [ ]:
def treated_control_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for t, label in [(0, "Control"), (1, "Treated")]:
        d = df.loc[df["T"] == t]
        rows.append({
            "group": label,
            "rows": len(d),
            "mean_sales": d["Y"].mean(),
            "median_sales": d["Y"].median(),
            "mean_discount": pd.to_numeric(d["discount"], errors="coerce").mean(),
            "mean_discount_pct": d["discount_pct"].mean(),
            "holiday_rate": d["holiday_flag"].mean(),
            "mean_lag_7": d["lag_7"].mean(),
            "mean_rolling_28": d["rolling_mean_28"].mean(),
            "unique_stores": d["store_id"].nunique(),
            "unique_products": d["product_id"].nunique(),
        })
    return pd.DataFrame(rows)

raw_group_summary = treated_control_summary(analytic_df)
display(raw_group_summary)

category_treatment = (
    analytic_df.groupby("first_category_id", observed=True)
    .agg(rows=("T", "size"), treatment_rate=("T", "mean"), mean_sales=("Y", "mean"))
    .sort_values("rows", ascending=False)
)
display(category_treatment.head(15))

store_treatment = (
    analytic_df.groupby("store_id", observed=True)
    .agg(rows=("T", "size"), treatment_rate=("T", "mean"), mean_sales=("Y", "mean"))
    .sort_values("rows", ascending=False)
)
display(store_treatment.head(15))

# Composition differences are descriptive associations, not adjustment results.
def composition_table(df: pd.DataFrame, col: str, top_n: int = 10) -> pd.DataFrame:
    top = df[col].value_counts().head(top_n).index
    rows = []
    for value in top:
        for tval, label in [(0, "Control"), (1, "Treated")]:
            d = df.loc[df["T"].eq(tval)]
            rows.append({
                "feature": col, "value": str(value), "group": label,
                "share_pct": float(d[col].eq(value).mean() * 100),
            })
    return pd.DataFrame(rows)

display(composition_table(analytic_df, "first_category_id"))
display(composition_table(analytic_df, "store_id"))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(raw_group_summary["group"], raw_group_summary["mean_sales"])
ax.set_ylabel("Mean observed sale_amount")
ax.set_title("Observed Sales by Treatment Group — Association Only")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "treated_control_sales_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Naive observed difference

In [ ]:
def naive_difference(df: pd.DataFrame) -> float:
    treated = df.loc[df["T"] == 1, "Y"]
    control = df.loc[df["T"] == 0, "Y"]
    if treated.empty or control.empty:
        raise ValueError("Naive difference requires both treated and control observations.")
    return float(treated.mean() - control.mean())

naive_observed_difference = naive_difference(analytic_df)
print(f"Naive Observed Difference: {naive_observed_difference:.6f}")
print("This is an association, not a causal effect estimate.")

## 10. Nuisance-model feature sets and category handling

In [ ]:
ID_CATEGORICALS = [
    "city_id", "store_id", "management_group_id", "first_category_id",
    "second_category_id", "third_category_id", "product_id",
]
BASE_NUMERIC_FEATURES = [
    "lag_1", "lag_7", "lag_14", "lag_28",
    "rolling_mean_7", "rolling_mean_28", "rolling_std_7", "rolling_std_28",
    "historical_zero_ratio_28",
    "historical_stockout_lag_1", "historical_stockout_lag_7",
    "day_of_week", "month", "is_weekend", "holiday_flag",
]
WEATHER_FEATURES = ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
PRIMARY_FEATURES = ID_CATEGORICALS + BASE_NUMERIC_FEATURES + WEATHER_FEATURES
NO_WEATHER_FEATURES = ID_CATEGORICALS + BASE_NUMERIC_FEATURES

# Same-day discount is intentionally absent from both lists.
assert "discount" not in PRIMARY_FEATURES and "discount_pct" not in PRIMARY_FEATURES
assert "Y" not in PRIMARY_FEATURES and "T" not in PRIMARY_FEATURES
assert "stock_hour6_22_cnt" not in PRIMARY_FEATURES

CATEGORY_LEVELS: Dict[str, pd.Index] = {}
for col in ID_CATEGORICALS:
    vals = pd.Index(analytic_df[col].dropna().unique())
    CATEGORY_LEVELS[col] = vals
    analytic_df[col] = pd.Categorical(analytic_df[col], categories=vals)


def feature_matrix(df: pd.DataFrame, feature_cols: Sequence[str]) -> pd.DataFrame:
    """Return a LightGBM-safe feature matrix with consistent categorical dtypes."""
    X = df.loc[:, list(feature_cols)].copy()
    for col in ID_CATEGORICALS:
        if col in X.columns:
            X[col] = pd.Categorical(X[col], categories=CATEGORY_LEVELS[col])
    return X

PRIMARY_FEATURES

A flexible LightGBM classifier is used for propensity estimation rather than dense one-hot logistic regression because store/product identifiers are high-cardinality. The treatment model uses **only measured pre-treatment covariates** and never sees the outcome or same-day discount.

## 11. Series-aware cross-fitting for propensity and outcome nuisance models

In [ ]:
PROPENSITY_PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    n_estimators=350,
    num_leaves=63,
    min_child_samples=80,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

OUTCOME_PARAMS = dict(
    objective="regression",
    learning_rate=0.05,
    n_estimators=400,
    num_leaves=63,
    min_child_samples=60,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)


def cross_fit_nuisance_models(
    df: pd.DataFrame,
    feature_cols: Sequence[str],
    n_splits: int = N_SPLITS,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Generate out-of-fold propensity, mu1, and mu0 predictions.

    Splits are grouped by complete store-product series so nuisance predictions for a row
    are produced by models that did not train on that row's series.
    """
    del random_state  # GroupKFold is deterministic; kept in signature for API consistency.
    out = df[["T", "Y", "_series_id", "dt"]].copy()
    out["propensity_oof"] = np.nan
    out["mu1_oof"] = np.nan
    out["mu0_oof"] = np.nan
    out["fold"] = -1

    splitter = GroupKFold(n_splits=n_splits)
    groups = df["_series_id"].to_numpy()

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(df, df["T"], groups=groups), start=1):
        train_part = df.iloc[train_idx]
        valid_part = df.iloc[valid_idx]

        if train_part["T"].nunique() != 2:
            raise RuntimeError(f"Fold {fold}: treatment model training data does not contain both classes.")
        if (train_part["T"] == 1).sum() < 100 or (train_part["T"] == 0).sum() < 100:
            raise RuntimeError(f"Fold {fold}: insufficient treated/control rows for outcome nuisance models.")

        X_train = feature_matrix(train_part, feature_cols)
        X_valid = feature_matrix(valid_part, feature_cols)

        prop = LGBMClassifier(**PROPENSITY_PARAMS)
        prop.fit(X_train, train_part["T"], categorical_feature=[c for c in ID_CATEGORICALS if c in feature_cols])

        treated_train = train_part["T"].eq(1).to_numpy()
        control_train = ~treated_train

        m1 = LGBMRegressor(**OUTCOME_PARAMS)
        m0 = LGBMRegressor(**OUTCOME_PARAMS)
        m1.fit(X_train.loc[treated_train], train_part.loc[treated_train, "Y"],
               categorical_feature=[c for c in ID_CATEGORICALS if c in feature_cols])
        m0.fit(X_train.loc[control_train], train_part.loc[control_train, "Y"],
               categorical_feature=[c for c in ID_CATEGORICALS if c in feature_cols])

        out.iloc[valid_idx, out.columns.get_loc("propensity_oof")] = prop.predict_proba(X_valid)[:, 1]
        out.iloc[valid_idx, out.columns.get_loc("mu1_oof")] = m1.predict(X_valid)
        out.iloc[valid_idx, out.columns.get_loc("mu0_oof")] = m0.predict(X_valid)
        out.iloc[valid_idx, out.columns.get_loc("fold")] = fold

        print(
            f"fold={fold} train_rows={len(train_idx):,} valid_rows={len(valid_idx):,} "
            f"train_series={train_part['_series_id'].nunique():,} valid_series={valid_part['_series_id'].nunique():,}"
        )
        del X_train, X_valid, prop, m1, m0, train_part, valid_part
        gc.collect()

    assert out[["propensity_oof", "mu1_oof", "mu0_oof"]].notna().all().all(), "Cross-fitting produced missing predictions."
    assert out["fold"].ge(1).all()
    assert out["propensity_oof"].between(0, 1).all()
    return out

primary_oof = cross_fit_nuisance_models(analytic_df, PRIMARY_FEATURES, n_splits=N_SPLITS)
for col in ["propensity_oof", "mu1_oof", "mu0_oof", "fold"]:
    analytic_df[col] = primary_oof[col].to_numpy()

print("Cross-fitting complete.")

## 12. Propensity diagnostics and positivity / overlap

In [ ]:
prop = analytic_df["propensity_oof"].astype(float)
t = analytic_df["T"].astype(int)
prop_auc = roc_auc_score(t, prop)

control_prop = prop[t.eq(0)]
treated_prop = prop[t.eq(1)]
common_support_low = max(control_prop.min(), treated_prop.min())
common_support_high = min(control_prop.max(), treated_prop.max())
common_support_mask = prop.between(common_support_low, common_support_high)
extreme_mask = (prop < PROPENSITY_EPS) | (prop > 1 - PROPENSITY_EPS)

propensity_diagnostics = pd.DataFrame([
    {"metric": "ROC_AUC", "value": prop_auc},
    {"metric": "propensity_min", "value": prop.min()},
    {"metric": "propensity_median", "value": prop.median()},
    {"metric": "propensity_max", "value": prop.max()},
    {"metric": "common_support_low", "value": common_support_low},
    {"metric": "common_support_high", "value": common_support_high},
    {"metric": "rows_below_eps", "value": int((prop < PROPENSITY_EPS).sum())},
    {"metric": "rows_above_1_minus_eps", "value": int((prop > 1 - PROPENSITY_EPS).sum())},
    {"metric": "extreme_propensity_pct", "value": float(extreme_mask.mean() * 100)},
    {"metric": "outside_common_support_rows", "value": int((~common_support_mask).sum())},
    {"metric": "outside_common_support_pct", "value": float((~common_support_mask).mean() * 100)},
])
display(propensity_diagnostics)
propensity_diagnostics.to_csv(TABLES_DIR / "propensity_diagnostics.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(control_prop, bins=50, alpha=0.55, density=True, label="T=0 control")
ax.hist(treated_prop, bins=50, alpha=0.55, density=True, label="T=1 treated")
ax.axvline(common_support_low, linestyle="--", linewidth=1, label="common support")
ax.axvline(common_support_high, linestyle="--", linewidth=1)
ax.set_xlabel("Out-of-fold propensity score")
ax.set_ylabel("Density")
ax.set_title("Propensity Score Overlap")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "propensity_overlap.png", dpi=160, bbox_inches="tight")
plt.show()

poor_overlap = (
    extreme_mask.mean() * 100 > POOR_OVERLAP_EXTREME_PCT
    or (~common_support_mask).mean() * 100 > POOR_OVERLAP_EXTREME_PCT
)
if poor_overlap:
    main_effect_mask = common_support_mask & ~extreme_mask
    print(
        "WARNING: overlap diagnostics indicate material positivity stress. "
        "The main adjusted estimand is restricted to common support and excludes extreme propensity rows."
    )
else:
    main_effect_mask = common_support_mask
    print("Overlap is not flagged as poor by the configured rule; common-support restriction is retained.")

print(f"Main effect sample rows: {main_effect_mask.sum():,} / {len(main_effect_mask):,}")
print(f"Rows removed: {(~main_effect_mask).sum():,}")

Restricting to overlap changes the target population: the adjusted ATE then applies to observations for which treated and control examples are empirically comparable. The notebook reports how many observations are excluded rather than hiding this change in estimand.

## 13. IPW weight diagnostics

In [ ]:
analysis_df = analytic_df.loc[main_effect_mask].copy()
# Only a machine-epsilon guard protects division by zero here; substantive clipping comes later.
analysis_df["propensity_safe"] = analysis_df["propensity_oof"].clip(1e-6, 1 - 1e-6)
analysis_df["ipw_weight_raw"] = np.where(
    analysis_df["T"].eq(1),
    1.0 / analysis_df["propensity_safe"],
    1.0 / (1.0 - analysis_df["propensity_safe"]),
)

weight_summary_raw = analysis_df["ipw_weight_raw"].describe(percentiles=[0.01, 0.5, 0.95, 0.99]).to_frame("raw_ipw_weight")
display(weight_summary_raw)
print("raw weights > 10:", int((analysis_df["ipw_weight_raw"] > 10).sum()))
print("raw weights > 20:", int((analysis_df["ipw_weight_raw"] > 20).sum()))
print("raw max weight  :", float(analysis_df["ipw_weight_raw"].max()))

w_lo, w_hi = analysis_df["ipw_weight_raw"].quantile([0.01, 0.99]).tolist()
analysis_df["ipw_weight"] = analysis_df["ipw_weight_raw"].clip(w_lo, w_hi)
print(f"1st–99th percentile weight clipping bounds: {w_lo:.4f}, {w_hi:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(analysis_df["ipw_weight_raw"].clip(upper=analysis_df["ipw_weight_raw"].quantile(0.995)), bins=60)
ax.set_xlabel("Raw ATE IPW weight (plot truncated at p99.5)")
ax.set_ylabel("Rows")
ax.set_title("IPW Weight Distribution Before 1st–99th Percentile Clipping")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ipw_weight_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

## 14. Covariate balance: standardized mean differences

In [ ]:
def weighted_mean(x: np.ndarray, w: np.ndarray) -> float:
    mask = np.isfinite(x) & np.isfinite(w)
    if mask.sum() == 0 or w[mask].sum() <= 0:
        return np.nan
    return float(np.sum(x[mask] * w[mask]) / np.sum(w[mask]))


def weighted_var(x: np.ndarray, w: np.ndarray) -> float:
    mask = np.isfinite(x) & np.isfinite(w)
    if mask.sum() == 0 or w[mask].sum() <= 0:
        return np.nan
    mu = weighted_mean(x[mask], w[mask])
    return float(np.sum(w[mask] * (x[mask] - mu) ** 2) / np.sum(w[mask]))


def standardized_mean_difference(
    df: pd.DataFrame,
    feature: str,
    weight_col: Optional[str] = None,
) -> float:
    x = pd.to_numeric(df[feature], errors="coerce").to_numpy(float)
    t = df["T"].to_numpy(int)
    w = np.ones(len(df), dtype=float) if weight_col is None else df[weight_col].to_numpy(float)

    x1, w1 = x[t == 1], w[t == 1]
    x0, w0 = x[t == 0], w[t == 0]
    m1, m0 = weighted_mean(x1, w1), weighted_mean(x0, w0)
    v1, v0 = weighted_var(x1, w1), weighted_var(x0, w0)
    pooled = math.sqrt(max((v1 + v0) / 2, 0)) if np.isfinite(v1) and np.isfinite(v0) else np.nan
    if not np.isfinite(pooled) or pooled == 0:
        return 0.0 if np.isclose(m1, m0, equal_nan=False) else np.nan
    return float((m1 - m0) / pooled)

BALANCE_FEATURES = BASE_NUMERIC_FEATURES + WEATHER_FEATURES
balance_rows = []
for f in BALANCE_FEATURES:
    balance_rows.append({
        "feature": f,
        "SMD_before": standardized_mean_difference(analysis_df, f),
        "SMD_after": standardized_mean_difference(analysis_df, f, "ipw_weight"),
    })
propensity_balance = pd.DataFrame(balance_rows)
propensity_balance["abs_SMD_before"] = propensity_balance["SMD_before"].abs()
propensity_balance["abs_SMD_after"] = propensity_balance["SMD_after"].abs()
propensity_balance.to_csv(TABLES_DIR / "propensity_balance.csv", index=False)
display(propensity_balance.sort_values("abs_SMD_before", ascending=False))

plot_bal = propensity_balance.sort_values("abs_SMD_before")
fig, ax = plt.subplots(figsize=(8, max(5, len(plot_bal) * 0.3)))
y = np.arange(len(plot_bal))
ax.scatter(plot_bal["SMD_before"], y, label="Before weighting")
ax.scatter(plot_bal["SMD_after"], y, label="After weighting")
ax.axvline(-0.10, linestyle="--", linewidth=1)
ax.axvline(0.10, linestyle="--", linewidth=1)
ax.axvline(0, linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(plot_bal["feature"])
ax.set_xlabel("Standardized Mean Difference")
ax.set_title("Covariate Balance (Love-Plot Style)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "covariate_balance_love_plot.png", dpi=160, bbox_inches="tight")
plt.show()

`|SMD| < 0.10` is used as a practical diagnostic target, not an absolute theorem. SMD is reported for numeric/binary pre-treatment covariates; arbitrary numeric codes for store/product/category IDs are **not** treated as meaningful continuous measurements for SMD.

## 15. ATE estimators: regression, IPW, and cross-fitted AIPW

In [ ]:
def aipw_score(df: pd.DataFrame, propensity_eps: float = PROPENSITY_EPS) -> np.ndarray:
    e = df["propensity_oof"].clip(propensity_eps, 1 - propensity_eps).to_numpy(float)
    t = df["T"].to_numpy(float)
    y = df["Y"].to_numpy(float)
    m1 = df["mu1_oof"].to_numpy(float)
    m0 = df["mu0_oof"].to_numpy(float)
    return (m1 - m0) + t * (y - m1) / e - (1 - t) * (y - m0) / (1 - e)


def regression_effect(df: pd.DataFrame) -> np.ndarray:
    return (df["mu1_oof"] - df["mu0_oof"]).to_numpy(float)


def ipw_hajek_ate(df: pd.DataFrame, weight_col: str = "ipw_weight") -> float:
    t = df["T"].eq(1)
    wt = df.loc[t, weight_col]
    wc = df.loc[~t, weight_col]
    if wt.sum() <= 0 or wc.sum() <= 0:
        raise ZeroDivisionError("IPW group weight denominator is zero.")
    mu1 = np.average(df.loc[t, "Y"], weights=wt)
    mu0 = np.average(df.loc[~t, "Y"], weights=wc)
    return float(mu1 - mu0)

analysis_df["aipw_score"] = aipw_score(analysis_df)
analysis_df["regression_effect"] = regression_effect(analysis_df)

ate_point = {
    "Naive": naive_difference(analysis_df),
    "Regression": float(analysis_df["regression_effect"].mean()),
    "IPW": ipw_hajek_ate(analysis_df),
    "AIPW": float(analysis_df["aipw_score"].mean()),
}
ate_point

### Why AIPW is doubly robust

AIPW combines a treatment-assignment model with treated/control outcome regressions. Under the standard identification assumptions, the estimator can remain consistent if **either** the propensity model or the outcome-regression component is correctly specified (with the required regularity conditions). Cross-fitting reduces overfitting bias by producing nuisance predictions out of sample at the series level. This property does not protect against unmeasured confounding or violations of positivity.

## 16. Series-cluster bootstrap confidence intervals

In [ ]:
def build_cluster_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate sufficient statistics by store-product series for efficient cluster bootstrap."""
    temp = df.copy()
    temp["treated_y"] = np.where(temp["T"].eq(1), temp["Y"], 0.0)
    temp["control_y"] = np.where(temp["T"].eq(0), temp["Y"], 0.0)
    temp["treated_n"] = temp["T"].astype(float)
    temp["control_n"] = 1.0 - temp["T"].astype(float)
    temp["wt_y"] = np.where(temp["T"].eq(1), temp["ipw_weight"] * temp["Y"], 0.0)
    temp["wc_y"] = np.where(temp["T"].eq(0), temp["ipw_weight"] * temp["Y"], 0.0)
    temp["wt"] = np.where(temp["T"].eq(1), temp["ipw_weight"], 0.0)
    temp["wc"] = np.where(temp["T"].eq(0), temp["ipw_weight"], 0.0)

    return temp.groupby("_series_id", observed=True).agg(
        rows=("Y", "size"),
        treated_y=("treated_y", "sum"),
        control_y=("control_y", "sum"),
        treated_n=("treated_n", "sum"),
        control_n=("control_n", "sum"),
        wt_y=("wt_y", "sum"),
        wc_y=("wc_y", "sum"),
        wt=("wt", "sum"),
        wc=("wc", "sum"),
        regression_sum=("regression_effect", "sum"),
        aipw_sum=("aipw_score", "sum"),
    ).reset_index(drop=True)


def bootstrap_ate_methods(cluster_summary: pd.DataFrame, n_bootstraps: int, seed: int = RANDOM_STATE) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n_clusters = len(cluster_summary)
    arr = cluster_summary.drop(columns=[]).to_dict("list")
    numeric = {k: np.asarray(v, dtype=float) for k, v in arr.items() if k != "_series_id"}
    rows = []

    for b in tqdm(range(n_bootstraps), desc="Cluster bootstrap", unit="rep"):
        idx = rng.integers(0, n_clusters, size=n_clusters)
        total_rows = numeric["rows"][idx].sum()
        t_n = numeric["treated_n"][idx].sum()
        c_n = numeric["control_n"][idx].sum()

        naive = np.nan
        if t_n > 0 and c_n > 0:
            naive = numeric["treated_y"][idx].sum() / t_n - numeric["control_y"][idx].sum() / c_n

        regression = numeric["regression_sum"][idx].sum() / total_rows

        ipw = np.nan
        wt = numeric["wt"][idx].sum(); wc = numeric["wc"][idx].sum()
        if wt > 0 and wc > 0:
            ipw = numeric["wt_y"][idx].sum() / wt - numeric["wc_y"][idx].sum() / wc

        aipw = numeric["aipw_sum"][idx].sum() / total_rows
        rows.extend([
            {"bootstrap": b, "method": "Naive", "estimate": naive},
            {"bootstrap": b, "method": "Regression", "estimate": regression},
            {"bootstrap": b, "method": "IPW", "estimate": ipw},
            {"bootstrap": b, "method": "AIPW", "estimate": aipw},
        ])
    return pd.DataFrame(rows)

cluster_summary = build_cluster_summary(analysis_df)
bootstrap_results = bootstrap_ate_methods(cluster_summary, BOOTSTRAP_REPS)
bootstrap_results.to_csv(TABLES_DIR / "bootstrap_results.csv", index=False)

ate_rows = []
for method, estimate in ate_point.items():
    b = bootstrap_results.loc[bootstrap_results["method"] == method, "estimate"].dropna()
    ate_rows.append({
        "Method": method,
        "ATE": estimate,
        "SE": b.std(ddof=1),
        "CI_Lower": b.quantile(0.025),
        "CI_Upper": b.quantile(0.975),
        "observations": len(analysis_df),
        "series_clusters": analysis_df["_series_id"].nunique(),
    })
ate_estimates = pd.DataFrame(ate_rows)
ate_estimates.to_csv(TABLES_DIR / "ate_estimates.csv", index=False)
display(ate_estimates)

The bootstrap resamples complete `(store_id, product_id)` series rather than individual rows. This is more defensible for repeated time-series observations because within-series rows are not treated as independent draws. The nuisance models are held fixed during this bootstrap, so intervals quantify cluster-resampling uncertainty conditional on the cross-fitted nuisance estimates rather than full model-refit uncertainty.

## 17. ATT — effect among actually promoted observations

In [ ]:
def att_dr_components(df: pd.DataFrame, propensity_eps: float = PROPENSITY_EPS) -> pd.DataFrame:
    e = df["propensity_oof"].clip(propensity_eps, 1 - propensity_eps).to_numpy(float)
    t = df["T"].to_numpy(float)
    y = df["Y"].to_numpy(float)
    m0 = df["mu0_oof"].to_numpy(float)
    score_num = t * (y - m0) - (1 - t) * (e / (1 - e)) * (y - m0)
    out = df[["_series_id", "T"]].copy()
    out["att_score_num"] = score_num
    return out

att_parts = att_dr_components(analysis_df)
att_estimate = float(att_parts["att_score_num"].mean() / analysis_df["T"].mean())

att_cluster = att_parts.groupby("_series_id", observed=True).agg(
    score_sum=("att_score_num", "sum"),
    treated_count=("T", "sum"),
    rows=("T", "size"),
).reset_index(drop=True)

rng = np.random.default_rng(RANDOM_STATE + 17)
att_boot = []
for b in tqdm(range(BOOTSTRAP_REPS), desc="ATT cluster bootstrap", unit="rep"):
    idx = rng.integers(0, len(att_cluster), size=len(att_cluster))
    score_sum = att_cluster["score_sum"].to_numpy()[idx].sum()
    treated_count = att_cluster["treated_count"].to_numpy()[idx].sum()
    total_rows = att_cluster["rows"].to_numpy()[idx].sum()
    if treated_count > 0 and total_rows > 0:
        p_t = treated_count / total_rows
        att_boot.append((score_sum / total_rows) / p_t)

att_boot = pd.Series(att_boot, name="ATT_bootstrap")
att_estimates = pd.DataFrame([{
    "Method": "Doubly Robust ATT",
    "ATT": att_estimate,
    "SE": att_boot.std(ddof=1),
    "CI_Lower": att_boot.quantile(0.025),
    "CI_Upper": att_boot.quantile(0.975),
    "treated_observations": int(analysis_df["T"].sum()),
    "observations": len(analysis_df),
}])
att_estimates.to_csv(TABLES_DIR / "att_estimates.csv", index=False)
display(att_estimates)

## 18. Heterogeneous treatment effects from cross-fitted AIPW scores

In [ ]:
def cluster_mean_ci(df: pd.DataFrame, score_col: str = "aipw_score") -> Tuple[float, float, float, float]:
    """Cluster-robust mean and normal-approximation CI using series-level score sums."""
    values = df[score_col].to_numpy(float)
    estimate = float(np.mean(values))
    centered = df[["_series_id", score_col]].copy()
    centered["centered"] = centered[score_col] - estimate
    cluster_sums = centered.groupby("_series_id", observed=True)["centered"].sum().to_numpy(float)
    n = len(df); g = len(cluster_sums)
    if g < 2 or n == 0:
        return estimate, np.nan, np.nan, np.nan
    se = math.sqrt((g / (g - 1)) * np.sum(cluster_sums ** 2)) / n
    return estimate, se, estimate - 1.96 * se, estimate + 1.96 * se


def heterogeneous_effect_table(
    df: pd.DataFrame,
    segment_col: str,
    min_rows: int = MIN_SEGMENT_ROWS,
    min_treated: int = MIN_SEGMENT_TREATED,
    min_control: int = MIN_SEGMENT_CONTROL,
) -> pd.DataFrame:
    rows = []
    for value, d in df.groupby(segment_col, observed=True, dropna=False):
        n = len(d); nt = int(d["T"].sum()); nc = n - nt
        if n < min_rows or nt < min_treated or nc < min_control:
            continue
        est, se, lo, hi = cluster_mean_ci(d)
        rows.append({
            "segment_type": segment_col,
            "segment": str(value),
            "sample_size": n,
            "treated_count": nt,
            "control_count": nc,
            "estimated_effect": est,
            "SE_cluster": se,
            "CI_Lower": lo,
            "CI_Upper": hi,
        })
    return pd.DataFrame(rows)

segment_columns = [
    "first_category_id", "store_id", "city_id", "demand_level",
    "intermittency_level", "is_weekend", "holiday_flag",
]
heterogeneous_tables = [heterogeneous_effect_table(analysis_df, col) for col in segment_columns]
heterogeneous_treatment_effects = pd.concat(
    [t for t in heterogeneous_tables if not t.empty], ignore_index=True
) if any(not t.empty for t in heterogeneous_tables) else pd.DataFrame()
heterogeneous_treatment_effects.to_csv(TABLES_DIR / "heterogeneous_treatment_effects.csv", index=False)
display(heterogeneous_treatment_effects.head(30))

## 19. Conditional treatment effects (T-learner interpretation)

In [ ]:
# Cross-fitted outcome predictions provide an honest, nuisance-model-based T-learner contrast.
analysis_df["cate_oof"] = analysis_df["mu1_oof"] - analysis_df["mu0_oof"]

cate_summary = analysis_df["cate_oof"].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]).to_frame("CATE_OOF")
display(cate_summary)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(analysis_df["cate_oof"].clip(
    lower=analysis_df["cate_oof"].quantile(0.005),
    upper=analysis_df["cate_oof"].quantile(0.995),
), bins=60)
ax.axvline(0, linewidth=1)
ax.set_xlabel("Cross-fitted T-learner CATE estimate")
ax.set_ylabel("Rows")
ax.set_title("Distribution of Estimated Conditional Promotion Effects")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cate_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


def cate_segment_table(df: pd.DataFrame, cols: Sequence[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        for value, d in df.groupby(col, observed=True, dropna=False):
            if len(d) < MIN_SEGMENT_ROWS:
                continue
            rows.append({
                "segment_type": col,
                "segment": str(value),
                "sample_size": len(d),
                "mean_cate": d["cate_oof"].mean(),
                "median_cate": d["cate_oof"].median(),
                "negative_cate_pct": (d["cate_oof"] < 0).mean() * 100,
            })
    return pd.DataFrame(rows)

cate_segment_analysis = cate_segment_table(analysis_df, ["first_category_id", "store_id", "product_id", "city_id"])
cate_segment_analysis.to_csv(TABLES_DIR / "cate_segment_analysis.csv", index=False)
display(cate_segment_analysis.sort_values("mean_cate", ascending=False).head(15))
display(cate_segment_analysis.sort_values("mean_cate", ascending=True).head(15))

These CATE values are model-based conditional effect estimates, not ground truth. Ranking can be useful for scientific heterogeneity analysis, but promotion decisions should not be made from CATE alone; profitability, stock constraints, and service objectives belong to the next inventory-decision phase.

## 20. Promotion-response ranking visualizations

In [ ]:
cat_effects = heterogeneous_treatment_effects.loc[
    heterogeneous_treatment_effects["segment_type"].eq("first_category_id")
].sort_values("estimated_effect")

if not cat_effects.empty:
    fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(cat_effects))))
    ax.barh(cat_effects["segment"], cat_effects["estimated_effect"])
    ax.axvline(0, linewidth=1)
    ax.set_xlabel("Cross-fitted AIPW effect estimate")
    ax.set_ylabel("First category")
    ax.set_title("Estimated Promotion Effect by Product Category")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "treatment_effect_by_category.png", dpi=160, bbox_inches="tight")
    plt.show()

rank_plot = cate_segment_analysis.loc[cate_segment_analysis["segment_type"].eq("first_category_id")].copy()
if not rank_plot.empty:
    rank_plot = pd.concat([rank_plot.nsmallest(5, "mean_cate"), rank_plot.nlargest(5, "mean_cate")]).drop_duplicates()
    rank_plot = rank_plot.sort_values("mean_cate")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(rank_plot["segment"], rank_plot["mean_cate"])
    ax.axvline(0, linewidth=1)
    ax.set_xlabel("Mean cross-fitted CATE")
    ax.set_ylabel("Category")
    ax.set_title("Top / Bottom Category Promotion Responders")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "top_bottom_responder_segments.png", dpi=160, bbox_inches="tight")
    plt.show()

## 21. Sensitivity analyses

In [ ]:
def aipw_estimate_from_oof(df: pd.DataFrame, eps: float = PROPENSITY_EPS) -> float:
    return float(np.mean(aipw_score(df, propensity_eps=eps)))

sensitivity_rows = []

# A: primary model with weather.
sensitivity_rows.append({
    "analysis": "A_with_weather_primary",
    "ATE_AIPW": aipw_estimate_from_oof(analysis_df),
    "rows": len(analysis_df),
    "notes": "Cross-fitted primary nuisance models with weather; primary overlap sample",
})

# B: full cross-fitting without weather.
print("Running no-weather cross-fitting sensitivity...")
no_weather_oof = cross_fit_nuisance_models(analytic_df, NO_WEATHER_FEATURES, n_splits=N_SPLITS)
no_weather_df = analytic_df.loc[main_effect_mask].copy()
for col in ["propensity_oof", "mu1_oof", "mu0_oof", "fold"]:
    no_weather_df[col] = no_weather_oof.loc[main_effect_mask, col].to_numpy()
sensitivity_rows.append({
    "analysis": "B_without_weather",
    "ATE_AIPW": aipw_estimate_from_oof(no_weather_df),
    "rows": len(no_weather_df),
    "notes": "Cross-fitted nuisance models excluding realized weather",
})

# C: propensity clipping thresholds, keeping nuisance predictions fixed.
for eps in (0.005, 0.01, 0.025, 0.05):
    sensitivity_rows.append({
        "analysis": f"C_propensity_clip_{eps:.3f}",
        "ATE_AIPW": aipw_estimate_from_oof(analysis_df, eps=eps),
        "rows": len(analysis_df),
        "notes": f"Primary OOF nuisance predictions; propensity clipped to [{eps}, {1-eps}]",
    })

# D: existing-series-only subgroup. Eval is used only for membership labeling, not outcomes.
existing_df = analysis_df.loc[analysis_df["existing_series"]].copy()
if len(existing_df) and existing_df["T"].nunique() == 2:
    sensitivity_rows.append({
        "analysis": "D_existing_series_only",
        "ATE_AIPW": aipw_estimate_from_oof(existing_df),
        "rows": len(existing_df),
        "notes": "Train observations from sampled series also present in eval; eval outcomes never used",
    })
else:
    sensitivity_rows.append({
        "analysis": "D_existing_series_only",
        "ATE_AIPW": np.nan,
        "rows": len(existing_df),
        "notes": "Not estimable: subgroup lacks adequate treated/control support",
    })

# Stockout-censoring sensitivity: prior-stockout-free subgroup only (pre-treatment history).
nostock_df = analysis_df.loc[analysis_df["recent_stockout_history"].eq(0)].copy()
if len(nostock_df) and nostock_df["T"].nunique() == 2:
    sensitivity_rows.append({
        "analysis": "stockout_history_no",
        "ATE_AIPW": aipw_estimate_from_oof(nostock_df),
        "rows": len(nostock_df),
        "notes": "Rows with no stockout at lag 1 or lag 7; observed sales may still be censored in other ways",
    })

causal_sensitivity_analysis = pd.DataFrame(sensitivity_rows)
causal_sensitivity_analysis.to_csv(TABLES_DIR / "causal_sensitivity_analysis.csv", index=False)
display(causal_sensitivity_analysis)

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(causal_sensitivity_analysis))))
plot_sens = causal_sensitivity_analysis.dropna(subset=["ATE_AIPW"]).sort_values("ATE_AIPW")
ax.barh(plot_sens["analysis"], plot_sens["ATE_AIPW"])
ax.axvline(0, linewidth=1)
ax.set_xlabel("AIPW ATE estimate")
ax.set_title("Causal Sensitivity Analysis")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sensitivity_analysis_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 22. Placebo / falsification test

In [ ]:
# A clearly pre-treatment outcome: sale amount seven observations before today's treatment.
# Today's treatment cannot logically cause Y_{t-7} under normal temporal ordering.
placebo_df = analytic_df.dropna(subset=["lag_7"]).copy()
placebo_df["Y_placebo"] = placebo_df["lag_7"].astype(float)


def cross_fit_placebo_outcomes(
    df: pd.DataFrame,
    feature_cols: Sequence[str],
    propensity_oof: pd.Series,
    n_splits: int = N_SPLITS,
) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["propensity_oof"] = propensity_oof.reindex(df.index).to_numpy()
    out["mu1_oof"] = np.nan
    out["mu0_oof"] = np.nan
    groups = df["_series_id"].to_numpy()
    splitter = GroupKFold(n_splits=n_splits)

    # To avoid directly feeding the placebo outcome back as lag_7, remove lag_7 from placebo outcome features.
    placebo_features = [f for f in feature_cols if f != "lag_7"]

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(df, df["T"], groups=groups), start=1):
        tr = df.iloc[train_idx]; va = df.iloc[valid_idx]
        Xtr = feature_matrix(tr, placebo_features); Xva = feature_matrix(va, placebo_features)
        tr_t = tr["T"].eq(1).to_numpy()
        m1 = LGBMRegressor(**OUTCOME_PARAMS)
        m0 = LGBMRegressor(**OUTCOME_PARAMS)
        cats = [c for c in ID_CATEGORICALS if c in placebo_features]
        m1.fit(Xtr.loc[tr_t], tr.loc[tr_t, "Y_placebo"], categorical_feature=cats)
        m0.fit(Xtr.loc[~tr_t], tr.loc[~tr_t, "Y_placebo"], categorical_feature=cats)
        out.iloc[valid_idx, out.columns.get_loc("mu1_oof")] = m1.predict(Xva)
        out.iloc[valid_idx, out.columns.get_loc("mu0_oof")] = m0.predict(Xva)
        del tr, va, Xtr, Xva, m1, m0
        gc.collect()
    return out

placebo_nuisance = cross_fit_placebo_outcomes(
    placebo_df,
    PRIMARY_FEATURES,
    analytic_df["propensity_oof"],
    n_splits=N_SPLITS,
)
placebo_eval = placebo_df[["T", "Y_placebo", "_series_id"]].copy()
placebo_eval["Y"] = placebo_eval["Y_placebo"]
placebo_eval["propensity_oof"] = placebo_nuisance["propensity_oof"].to_numpy()
placebo_eval["mu1_oof"] = placebo_nuisance["mu1_oof"].to_numpy()
placebo_eval["mu0_oof"] = placebo_nuisance["mu0_oof"].to_numpy()
placebo_eval = placebo_eval.loc[placebo_eval.index.intersection(analysis_df.index)].copy()

placebo_eval["aipw_score"] = aipw_score(placebo_eval)
placebo_effect, placebo_se, placebo_lo, placebo_hi = cluster_mean_ci(placebo_eval)
placebo_concern = not (placebo_lo <= 0 <= placebo_hi) if np.isfinite(placebo_lo) and np.isfinite(placebo_hi) else True
placebo_test_results = pd.DataFrame([{
    "placebo_outcome": "lag_7 pre-treatment sales",
    "AIPW_placebo_effect": placebo_effect,
    "SE_cluster": placebo_se,
    "CI_Lower": placebo_lo,
    "CI_Upper": placebo_hi,
    "observations": len(placebo_eval),
    "residual_confounding_concern": placebo_concern,
}])
placebo_test_results.to_csv(TABLES_DIR / "placebo_test_results.csv", index=False)
display(placebo_test_results)
if placebo_concern:
    print("PLACEBO WARNING: the 95% cluster-robust interval excludes zero; residual confounding or misspecification deserves attention.")
else:
    print("The placebo interval includes zero under this diagnostic; this is reassuring but does not prove ignorability.")

A strong non-zero placebo effect would be evidence that residual confounding, treatment timing, or model misspecification may remain. A weak placebo effect does **not** prove that all unmeasured confounding has been removed.

## 23. Stockout censoring analysis

In [ ]:
stockout_compare = []
for value, label in [(0, "No recent stockout history"), (1, "Recent stockout history")]:
    d = analysis_df.loc[analysis_df["recent_stockout_history"].eq(value)]
    if len(d) >= MIN_SEGMENT_ROWS and d["T"].nunique() == 2:
        est, se, lo, hi = cluster_mean_ci(d)
        stockout_compare.append({
            "group": label,
            "rows": len(d),
            "ATE_AIPW": est,
            "CI_Lower": lo,
            "CI_Upper": hi,
            "mean_observed_sales": d["Y"].mean(),
        })
stockout_sensitivity = pd.DataFrame(stockout_compare)
display(stockout_sensitivity)

Observed sales can be below latent demand when inventory is unavailable. Restricting to rows without lagged stockout history is only a sensitivity check; it does **not** recover unobserved latent demand and it does not guarantee that current outcomes are uncensored.

## 24. ATE method comparison visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_ate = ate_estimates.copy()
ax.errorbar(
    plot_ate["Method"],
    plot_ate["ATE"],
    yerr=[plot_ate["ATE"] - plot_ate["CI_Lower"], plot_ate["CI_Upper"] - plot_ate["ATE"]],
    fmt="o",
    capsize=4,
)
ax.axhline(0, linewidth=1)
ax.set_ylabel("Estimated change in observed sale_amount")
ax.set_title("Promotion Effect Estimates by Method")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ate_method_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 25. Fit final nuisance models for reusable causal artifacts

In [ ]:
X_full = feature_matrix(analysis_df, PRIMARY_FEATURES)
cats = [c for c in ID_CATEGORICALS if c in PRIMARY_FEATURES]

final_propensity_model = LGBMClassifier(**PROPENSITY_PARAMS)
final_propensity_model.fit(X_full, analysis_df["T"], categorical_feature=cats)

final_outcome_treated = LGBMRegressor(**OUTCOME_PARAMS)
final_outcome_control = LGBMRegressor(**OUTCOME_PARAMS)
mask_t = analysis_df["T"].eq(1)
final_outcome_treated.fit(X_full.loc[mask_t], analysis_df.loc[mask_t, "Y"], categorical_feature=cats)
final_outcome_control.fit(X_full.loc[~mask_t], analysis_df.loc[~mask_t, "Y"], categorical_feature=cats)

final_propensity_model.booster_.save_model(str(CAUSAL_MODELS_DIR / "propensity_model.txt"))
final_outcome_treated.booster_.save_model(str(CAUSAL_MODELS_DIR / "outcome_treated.txt"))
final_outcome_control.booster_.save_model(str(CAUSAL_MODELS_DIR / "outcome_control.txt"))
print("Saved final nuisance models under models/causal/.")

## 26. Final findings generated from executed results

In [ ]:
main_aipw = float(ate_estimates.loc[ate_estimates["Method"].eq("AIPW"), "ATE"].iloc[0])
main_aipw_lo = float(ate_estimates.loc[ate_estimates["Method"].eq("AIPW"), "CI_Lower"].iloc[0])
main_aipw_hi = float(ate_estimates.loc[ate_estimates["Method"].eq("AIPW"), "CI_Upper"].iloc[0])
att_value = float(att_estimates["ATT"].iloc[0])

max_before = propensity_balance["abs_SMD_before"].max()
max_after = propensity_balance["abs_SMD_after"].max()
balanced_after_pct = (propensity_balance["abs_SMD_after"] < 0.10).mean() * 100

sens_valid = causal_sensitivity_analysis.dropna(subset=["ATE_AIPW"])
sens_range = (sens_valid["ATE_AIPW"].min(), sens_valid["ATE_AIPW"].max()) if len(sens_valid) else (np.nan, np.nan)

print("# Findings\n")
print(f"1. Naive promotion-sales difference: {naive_observed_difference:.6f} (association only).")
print(f"2. Cross-fitted AIPW ATE on the analysis population: {main_aipw:.6f} "
      f"with cluster-bootstrap 95% CI [{main_aipw_lo:.6f}, {main_aipw_hi:.6f}].")
print(f"3. Doubly robust ATT among promoted observations: {att_value:.6f}.")
ipw_value = float(ate_estimates.loc[ate_estimates["Method"].eq("IPW"), "ATE"].iloc[0])
print(f"4. IPW vs AIPW difference: {ipw_value - main_aipw:+.6f}; agreement should be judged relative to the effect scale and uncertainty.")
print("5. Method comparison:")
display(ate_estimates[["Method", "ATE", "CI_Lower", "CI_Upper", "SE"]])
ci_width = main_aipw_hi - main_aipw_lo
print(f"6. AIPW 95% CI width: {ci_width:.6f}; zero {'is inside' if main_aipw_lo <= 0 <= main_aipw_hi else 'is outside'} the interval.")
print(f"7. Propensity ROC-AUC: {prop_auc:.4f}. Extreme propensity share: {extreme_mask.mean()*100:.2f}%. "
      f"Rows outside common support: {(~common_support_mask).mean()*100:.2f}%.")
print(f"8. Maximum absolute SMD changed from {max_before:.3f} before weighting to {max_after:.3f} after weighting; "
      f"{balanced_after_pct:.1f}% of reported numeric/binary covariates are below |SMD|=0.10 after weighting.")
print(f"9. Sensitivity AIPW estimates span [{sens_range[0]:.6f}, {sens_range[1]:.6f}] across executed analyses.")
print(f"10. Placebo AIPW effect on pre-treatment lag_7 sales: {placebo_effect:.6f}, "
      f"95% CI [{placebo_lo:.6f}, {placebo_hi:.6f}]; concern flag={placebo_concern}.")
if not nostock_df.empty and nostock_df["T"].nunique() == 2:
    nostock_effect = aipw_estimate_from_oof(nostock_df)
    print(f"11. No-recent-stockout-history AIPW estimate: {nostock_effect:.6f}; difference from primary={nostock_effect-main_aipw:+.6f}.")

if not heterogeneous_treatment_effects.empty:
    reliable = heterogeneous_treatment_effects.sort_values("estimated_effect")
    print("12. Lowest reliable segment effects:")
    display(reliable.head(10))
    print("13. Highest reliable segment effects:")
    display(reliable.tail(10).sort_values("estimated_effect", ascending=False))
else:
    print("12-13. No segments met the configured minimum treated/control/sample-size rule.")

print("\nInterpretation must remain conditional on the causal assumptions and the stockout-censored observed-sales outcome.")

## 27. Causal assumptions and limitations

Causal interpretation requires, at minimum:

1. **Consistency:** the observed outcome under the realized treatment corresponds to the relevant potential outcome, and “promotion” is sufficiently well-defined.
2. **Positivity:** for covariate patterns in the target population, both treatment and control have non-zero probability.
3. **No major unmeasured confounding:** after adjustment, no important omitted pre-treatment variable jointly drives promotion assignment and sales.
4. **Correct temporal ordering:** every adjustment feature is measured or known before treatment/outcome realization.
5. **No problematic interference:** one store-product observation's treatment does not materially change another unit's outcome in ways ignored by the estimand.
6. **Reasonable nuisance estimation:** propensity and outcome models capture enough relevant structure for IPW/AIPW behavior to be stable.

Additional limitations:

- `sale_amount` is normalized and represents observed sales, not necessarily physical-unit latent demand.
- Stockouts can censor demand; this notebook does not recover latent demand.
- Promotion variables are observational, so residual/unmeasured confounding can remain.
- Same-day discount is plausibly a mediator and is intentionally excluded from the primary total-effect adjustment set.
- Weather is assumed exogenous to promotion; sensitivity analysis removes it.
- Grouped cross-fitting reduces within-series nuisance overfitting but does not solve unmeasured confounding.
- Cluster bootstrap intervals hold fitted nuisance predictions fixed rather than refitting every ML model in every bootstrap replicate.
- CATE/T-learner estimates are model-dependent and should not be treated as individual causal truths.
- Existing-series-only sensitivity uses eval **membership only** and therefore targets a selected subgroup; no eval outcomes enter estimation.
- 30% deterministic complete-series sampling limits evidence about full-dataset scalability and may affect precision.
- Results from this retailer may not generalize to unrelated retailers, assortments, or promotion policies.
- Profitability and inventory-optimal treatment decisions require costs, margins, service-level objectives, and operational constraints that are deliberately deferred.

## 28. Programmatic final quality and artifact audit

In [ ]:
# Required scientific checks.
quality_checks = {
    "no_current_day_sales_in_features": "Y" not in PRIMARY_FEATURES and "sale_amount" not in PRIMARY_FEATURES,
    "no_same_day_stock_in_features": "stock_hour6_22_cnt" not in PRIMARY_FEATURES,
    "discount_not_primary_confounder": "discount" not in PRIMARY_FEATURES and "discount_pct" not in PRIMARY_FEATURES,
    "treatment_binary": set(analytic_df["T"].unique()).issubset({0, 1}),
    "propensity_valid": analytic_df["propensity_oof"].between(0, 1).all(),
    "cross_fitting_complete": analytic_df[["propensity_oof", "mu1_oof", "mu0_oof"]].notna().all().all(),
    "overlap_diagnostics_exist": (TABLES_DIR / "propensity_diagnostics.csv").exists(),
    "balance_before_after_exist": (TABLES_DIR / "propensity_balance.csv").exists(),
    "ate_and_att_separate": "ATE" in ate_estimates.columns and "ATT" in att_estimates.columns,
    "bootstrap_executed": len(bootstrap_results) >= 4 * BOOTSTRAP_REPS,
    "placebo_completed": len(placebo_test_results) == 1,
    "sensitivity_completed": len(causal_sensitivity_analysis) >= 4,
}
quality_audit = pd.DataFrame({"check": list(quality_checks.keys()), "passed": list(quality_checks.values())})
display(quality_audit)
assert quality_audit["passed"].all(), "One or more final causal quality checks failed."

mandatory_artifacts = [
    TABLES_DIR / "causal_data_audit.csv",
    TABLES_DIR / "propensity_balance.csv",
    TABLES_DIR / "propensity_diagnostics.csv",
    TABLES_DIR / "ate_estimates.csv",
    TABLES_DIR / "att_estimates.csv",
    TABLES_DIR / "bootstrap_results.csv",
    TABLES_DIR / "heterogeneous_treatment_effects.csv",
    TABLES_DIR / "cate_segment_analysis.csv",
    TABLES_DIR / "causal_sensitivity_analysis.csv",
    TABLES_DIR / "placebo_test_results.csv",
]

artifact_audit = pd.DataFrame([
    {"artifact": str(p), "exists": p.exists(), "size_bytes": p.stat().st_size if p.exists() else 0}
    for p in mandatory_artifacts
])
display(artifact_audit)
assert artifact_audit["exists"].all(), "One or more mandatory output artifacts are missing."

print("Notebook 03 complete.\n")
print("Completed:")
print("✓ Treatment definition")
print("✓ Propensity modeling")
print("✓ Balance diagnostics")
print("✓ IPW")
print("✓ Doubly robust AIPW")
print("✓ ATE / ATT")
print("✓ CATE")
print("✓ Bootstrap uncertainty")
print("✓ Sensitivity analysis")
print("✓ Placebo testing\n")
print("Next:")
print("Notebook 04 — Inventory Decision Optimization")